<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_5_adaptacion.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_5_adaptacion.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_5_adaptacion.ipynb)

# SI7011 · Sesión 06 · 5. Un BERT preentrenado: congelar, ajustar o LoRA
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña las **Partes 3 y 5** de las diapositivas. Primero miramos qué aprendió DistilBERT en el preentrenamiento (predicción enmascarada, diapositiva 41). Después lo usamos de tres formas para clasificar reseñas de IMDB con **solo 2 000 ejemplos**, contra una LSTM entrenada desde cero con la receta de la sesión 05. Todo con las herramientas de Hugging Face que se usan en la práctica: `pipeline`, `datasets`, `Trainer` y `peft`.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | DistilBERT completa `[MASK]` | qué sabe sin haber visto IMDB |
| 2 | "bank" en tres frases | la entrada es la misma; la salida depende del contexto |
| 3 | IMDB: 2 000 reseñas para entrenar | cuántas se recortan a 256 tokens |
| 4 | LSTM desde cero | la referencia |
| 5 | encoder congelado + sonda | una regresión logística: 769 parámetros |
| 6 | ajuste completo | los 66 millones |
| 7 | LoRA en $W_Q$ y $W_V$, más la cabeza | un 1 % de los parámetros |
| 8 | la tabla | exactitud, parámetros, tiempo y memoria |

Las estrategias usan **los mismos tokens y los mismos datos**, y las tres que entrenan con gradiente usan el **mismo `Trainer`**: lo único que cambia es qué parámetros entrenan (diapositiva 62).

**Usa GPU** (Kaggle T4, unos 10 minutos en total). En Kaggle activa **Internet** en *Settings*: DistilBERT e IMDB se bajan de Hugging Face.

In [ ]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')     # Kaggle "GPU T4 x2": una sola GPU (SFTTrainer no admite DataParallel)
# Kaggle trae torchao 0.10, que las versiones recientes de transformers y peft rechazan al importarse
# ("Found an incompatible version of torchao"). Este notebook no lo usa: si es viejo, se desinstala.
import os
from importlib.metadata import version, PackageNotFoundError
try:
    if tuple(int(x) for x in version('torchao').split('.')[:2]) < (0, 16):
        os.system('pip uninstall -y -q torchao')
except (PackageNotFoundError, ValueError):
    pass

import json, random, textwrap, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

try:
    import peft
except ImportError:
    os.system('pip install -q peft')
    import peft
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

random.seed(0); torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '· peft', peft.__version__, '·', dev)
if dev == 'cpu':
    print('Sin GPU: el notebook usa pocos ejemplos. En Kaggle: Settings → Accelerator → GPU T4.')
MODELO = 'distilbert-base-uncased'      # 6 bloques, d = 768, preentrenado con predicción enmascarada

## 1. Lo que aprendió en el preentrenamiento
DistilBERT es un encoder de 6 bloques, $d = 768$, preentrenado en inglés (libros y Wikipedia) **ocultando el 15 % de los tokens y prediciéndolos con el contexto de ambos lados** (diapositivas 41–42). Nadie le dio etiquetas. Pidámosle que complete `[MASK]` con un `pipeline` de `fill-mask`.

**Predice antes de correr:** ¿sabe qué adjetivos van con *movie*? ¿Sabe geografía?

In [ ]:
completar = pipeline('fill-mask', model=MODELO, device=0 if dev == 'cuda' else -1)
for frase in ['the movie was absolutely [MASK].',
              'i fell asleep because the film was so [MASK].',
              'the acting was great, but the [MASK] was terrible.',
              'paris is the capital of [MASK].']:
    print(f'{frase:<52}', ' · '.join(f"{r['token_str']} {r['score']:.2f}" for r in completar(frase, top_k=5)))

Para predecir la palabra oculta tuvo que aprender gramática (después de *was so* va un adjetivo), algo del sentimiento (*fell asleep* va con *boring*) y hechos que el texto repite. Es lo que la diapositiva 49 llama "lo que aprende el preentrenamiento": todo eso está guardado en sus pesos antes de ver una sola reseña.

## 2. La misma palabra, tres vectores
La figura del comienzo de la sesión (F01), con un modelo entrenado: *bank* en un río, en un banco donde se deposita dinero y en un banco que sube las tasas. A la entrada, *bank* es la **misma fila** de la tabla $E$ en las tres frases; a la salida del último bloque, la autoatención ya mezcló el contexto.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODELO)
enc = AutoModel.from_pretrained(MODELO).to(dev).eval()

@torch.no_grad()
def vector(frase, palabra):
    """Entrada (embedding de token) y salida (último bloque) de `palabra` dentro de `frase`."""
    x = tok(frase, return_tensors='pt').to(dev)
    i = x['input_ids'][0].tolist().index(tok.convert_tokens_to_ids(palabra))
    return enc.get_input_embeddings()(x['input_ids'])[0, i], enc(**x).last_hidden_state[0, i]

frases = ['i sat on the bank of the river.', 'i deposited money at the bank.', 'the bank raised its interest rates.']
v = [vector(f, 'bank') for f in frases]
cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
print('"bank" en tres frases · similitud coseno')
print(f'  entrada (tabla E):  río–dinero {cos(v[0][0], v[1][0]):.2f} · dinero–intereses {cos(v[1][0], v[2][0]):.2f}')
print(f'  salida  (bloque 6): río–dinero {cos(v[0][1], v[1][1]):.2f} · dinero–intereses {cos(v[1][1], v[2][1]):.2f}')

A la entrada la similitud es exactamente 1. A la salida, los dos usos financieros quedan más cerca entre sí que del río. Por eso tiene sentido usar las **salidas** de BERT como representación (sección 5) y no los embeddings de entrada.

## 3. Los datos
IMDB: reseñas de películas en inglés, positivas o negativas (la sesión 05). Tomamos solo **2 000** para entrenar, 500 para validar y 2 000 de test: con pocas etiquetas es donde el preentrenamiento debería notarse. Los tokens son los de DistilBERT (WordPiece, minúsculas, $\lvert V\rvert \approx 30\,000$) para **todos** los modelos, también la LSTM.

In [ ]:
imdb = load_dataset('stanfordnlp/imdb')
N_TRAIN, N_VAL, N_TEST = (2000, 500, 2000) if dev == 'cuda' else (64, 32, 64)
mezcla = imdb['train'].shuffle(seed=0)
partes = {'train': mezcla.select(range(N_TRAIN)), 'val': mezcla.select(range(N_TRAIN, N_TRAIN + N_VAL)),
          'test': imdb['test'].shuffle(seed=0).select(range(N_TEST))}
MAX_LEN = 256 if dev == 'cuda' else 128

def tokenizar(lote):
    return tok([t.replace('<br />', ' ') for t in lote['text']], truncation=True, max_length=MAX_LEN)

D = {k: p.map(tokenizar, batched=True, remove_columns=['text']) for k, p in partes.items()}
for k, p in partes.items():
    print(f'{k}: {len(p)} reseñas, {np.mean(list(p["label"])):.0%} positivas')
largos = [len(x) for x in tok(list(partes['train']['text']))['input_ids']]
print(f'tokens por reseña: mediana {int(np.median(largos))} · {np.mean(np.array(largos) > MAX_LEN):.0%} se recortan a {MAX_LEN}')
print(tok.convert_ids_to_tokens(D['train'][0]['input_ids'][:24]))

`[CLS]` va al comienzo de cada reseña: su salida es la que usa la cabeza de clasificación (diapositiva 64). Las reseñas largas se recortan a 256 tokens: el tokenizador y el largo máximo son parte del contrato del modelo (diapositiva 75).

## 4. La referencia: una LSTM desde cero
`entrenar` arma un `Trainer` con los `TrainingArguments` de siempre (AdamW, warmup y coseno, clipping, precisión mixta, evaluación por época) y lo usan los tres modelos que se entrenan con gradiente. `DataCollatorWithPadding` rellena cada lote hasta su reseña más larga. Primero la LSTM bidireccional de la sesión 05, sin preentrenamiento: es un `nn.Module` propio, y basta con que devuelva la pérdida y los logits para que el `Trainer` lo entrene.

In [ ]:
colador = DataCollatorWithPadding(tok)            # rellena cada lote hasta su reseña más larga

def exactitud(pred):
    return {'accuracy': float((pred.predictions.argmax(-1) == pred.label_ids).mean())}

def entrenar(nombre, modelo, lr, epocas, B=16):
    """El mismo Trainer para todos: lo único que cambia entre estrategias es qué parámetros entrenan."""
    args = TrainingArguments(output_dir=f'salidas/{nombre}', learning_rate=lr, num_train_epochs=epocas,
                             per_device_train_batch_size=B, per_device_eval_batch_size=64, warmup_steps=0.1,
                             lr_scheduler_type='cosine', weight_decay=0.01, max_grad_norm=1.0,
                             fp16=dev == 'cuda', eval_strategy='epoch', save_strategy='no', logging_strategy='epoch',
                             report_to='none', seed=0)
    trainer = Trainer(model=modelo, args=args, train_dataset=D['train'], eval_dataset=D['val'],
                      data_collator=colador, compute_metrics=exactitud)
    if dev == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    trainer.train()
    segundos = time.time() - t0
    return fila(nombre, trainer, segundos, sum(p.numel() for p in modelo.parameters() if p.requires_grad),
                sum(p.numel() for p in modelo.parameters()))

def fila(nombre, trainer, segundos, entrenables, total):
    acc_val = trainer.evaluate()['eval_accuracy']
    pred = trainer.predict(D['test'])
    PRED[nombre] = pred.predictions.argmax(-1)
    return {'modelo': nombre, 'acc_val': acc_val, 'acc_test': pred.metrics['test_accuracy'],
            'entrenables': entrenables, 'total': total, 'segundos': segundos,
            'memoria_gb': torch.cuda.max_memory_allocated() / 2**30 if dev == 'cuda' else float('nan')}

PRED, R = {}, []
Y = {k: np.array(list(p['label'])) for k, p in partes.items()}
Y_TEST = Y['test']

In [ ]:
class LSTMTexto(nn.Module):
    """La receta de la sesión 05, con los mismos tokens: embedding, LSTM bidireccional, promedio con máscara.
    Devuelve lo que el Trainer espera: la pérdida y los logits."""
    def __init__(self, V, d=128, h=128):
        super().__init__()
        self.E = nn.Embedding(V, d, padding_idx=tok.pad_token_id)
        self.lstm = nn.LSTM(d, h, batch_first=True, bidirectional=True)
        self.salida = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * h, 2))

    def forward(self, input_ids, attention_mask, labels=None):
        H, _ = self.lstm(self.E(input_ids))
        m = attention_mask.unsqueeze(-1).to(H.dtype)
        logits = self.salida((H * m).sum(1) / m.sum(1))
        loss = F.cross_entropy(logits.float(), labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

torch.manual_seed(0)
R.append(entrenar('LSTM desde cero', LSTMTexto(len(tok)), lr=2e-3, epocas=8, B=32))
print(R[-1])

## 5. Encoder congelado + sonda
Diapositiva 64: pasamos los datos **una vez** por el encoder, guardamos $\mathbf h$ (la salida de `[CLS]`) y ajustamos una regresión logística de scikit-learn sobre esos vectores, como las sondas de la sesión 04. El encoder no recibe gradiente.

**Predice** (diapositiva 76): ordena las cuatro estrategias por exactitud y por parámetros entrenables. ¿Coinciden los dos órdenes?

In [ ]:
@torch.no_grad()
def features(parte, B=64):
    """La salida de [CLS] del encoder congelado, una sola vez por reseña (diapositiva 64)."""
    hs = []
    for i in range(0, len(D[parte]), B):
        lote = colador([{k: D[parte][j][k] for k in ('input_ids', 'attention_mask')}
                        for j in range(i, min(i + B, len(D[parte])))]).to(dev)
        with torch.autocast('cuda', dtype=torch.float16, enabled=dev == 'cuda'):
            hs.append(enc(**lote).last_hidden_state[:, 0].float().cpu())
    return torch.cat(hs).numpy()

if dev == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()
H = {k: features(k) for k in ('train', 'val', 'test')}
sonda = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))   # la cabeza: una regresión logística
sonda.fit(H['train'], Y['train'])
PRED['DistilBERT congelado + sonda'] = sonda.predict(H['test'])
R.append({'modelo': 'DistilBERT congelado + sonda', 'acc_val': sonda.score(H['val'], Y['val']),
          'acc_test': float((PRED['DistilBERT congelado + sonda'] == Y_TEST).mean()),
          'entrenables': H['train'].shape[1] + 1, 'total': enc.num_parameters() + H['train'].shape[1] + 1,
          'segundos': time.time() - t0,
          'memoria_gb': torch.cuda.max_memory_allocated() / 2**30 if dev == 'cuda' else float('nan')})
print(R[-1])

## 6. Ajuste completo
Diapositiva 65: `AutoModelForSequenceClassification` pone una cabeza nueva sobre DistilBERT y todo entrena, con la tasa de aprendizaje **pequeña** de la sesión 03 ($3 \cdot 10^{-5}$), warmup y pocas épocas. La cabeza empieza al azar; el warmup evita que sus primeros gradientes grandes desordenen el encoder.

In [ ]:
torch.manual_seed(0)
modelo = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)     # todo entrenable
R.append(entrenar('DistilBERT ajustado completo', modelo, lr=3e-5, epocas=3))
print(R[-1])
del modelo

## 7. LoRA
Diapositiva 66, con `peft`: `get_peft_model` congela el modelo y agrega $BA_r$ con $r = 8$ a $W_Q$ y $W_V$ (`q_lin`, `v_lin`) de los 6 bloques. La cabeza (`pre_classifier` y `classifier`) es nueva y también entrena: `modules_to_save`. LoRA son 147 456 valores; con la cabeza, un 1 % del modelo. Como $B$ empieza en cero, el primer paso parte exactamente del modelo preentrenado. La tasa es más alta que en el ajuste completo: se entrenan pocos parámetros, que empiezan en cero.

In [ ]:
torch.manual_seed(0)
base = AutoModelForSequenceClassification.from_pretrained(MODELO, num_labels=2)
cfg_lora = LoraConfig(task_type='SEQ_CLS', r=8, lora_alpha=16, lora_dropout=0.1,
                      target_modules=['q_lin', 'v_lin'],                    # W_Q y W_V de los 6 bloques
                      modules_to_save=['pre_classifier', 'classifier'])     # la cabeza nueva sí entrena
modelo = get_peft_model(base, cfg_lora)
modelo.print_trainable_parameters()
R.append(entrenar('DistilBERT + LoRA', modelo, lr=5e-4, epocas=3))
print(R[-1])

## 8. La tabla
Exactitud en **test** (se mira una sola vez, al final), parámetros entrenables, tiempo de entrenamiento y memoria pico de la GPU (diapositiva 75).

In [ ]:
tabla = pd.DataFrame(R).set_index('modelo')
tabla['% entrenable'] = 100 * tabla['entrenables'] / tabla['total']
print(tabla[['acc_val', 'acc_test', 'entrenables', '% entrenable', 'segundos', 'memoria_gb']].round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 4))
for nombre, r in tabla.iterrows():
    ax.scatter(r['entrenables'], r['acc_test'], s=70)
    ax.annotate(nombre, (r['entrenables'], r['acc_test']), xytext=(6, 4), textcoords='offset points', fontsize=9)
ax.set_xscale('log'); ax.set_xlabel('parámetros entrenables (escala log)'); ax.set_ylabel('exactitud en test')
plt.tight_layout(); plt.show()

Lo esperable (contrástalo con tu tabla): las tres variantes de DistilBERT superan a la LSTM con 2 000 reseñas; el ajuste completo y LoRA quedan cerca; la sonda queda detrás, con **muchísimos** menos parámetros entrenables y el menor tiempo. Los dos órdenes no coinciden: la estrategia más exacta no es la más barata, y la decisión depende de cuántas etiquetas, cuánta memoria y cuántas tareas haya (diapositiva 77).

Compara también con la sesión 05 (notebook 5.4), que entrenó con **20 000** reseñas: bolsa de embeddings 85.4 %, CNN 84.1 %, LSTM 82.6 %. Con la décima parte de las etiquetas, ¿cuáles de estos modelos las superan?

¿Dónde se equivoca la sonda y el ajuste completo no?

In [ ]:
a, b = 'DistilBERT congelado + sonda', 'DistilBERT ajustado completo'
idx = [i for i in range(N_TEST) if PRED[b][i] == Y_TEST[i] != PRED[a][i]]
print(f'{len(idx)} reseñas de test que el ajuste completo acierta y la sonda no. Tres:\n')
for i in idx[:3]:
    print('●', 'positiva' if Y_TEST[i] else 'negativa', '·', textwrap.shorten(partes['test'][i]['text'], 350), '\n')

In [ ]:
json.dump({'modelo_base': MODELO, 'n_train': N_TRAIN, 'n_test': N_TEST, 'max_len': MAX_LEN, 'filas': R},
          open('resultados_adaptacion.json', 'w'), ensure_ascii=False, indent=1)
print('guardado: resultados_adaptacion.json')

## Tu turno · 10 minutos
1. **Menos etiquetas.** Repite con `N_TRAIN = 200`. ¿Cuál estrategia pierde más? ¿Y la LSTM? (Es la curva de eficiencia en etiquetas de la sesión 04.)
2. **El rango.** LoRA con `r = 2` y `r = 32`. ¿Cuántos parámetros entrenables? ¿Cambia la exactitud?
3. **Sin warmup.** En `entrenar`, pon `warmup_steps=0` y corre el ajuste completo con `lr=1e-4`. ¿Qué le pasa a la validación en la primera época?
4. **Otra salida.** En la sonda, usa el promedio de todas las salidas (con la máscara de relleno) en lugar de `[CLS]`. ¿Mejora?

**Para pensar.** Una clínica tiene 800 notas clínicas en español y una GPU de 16 GB (la pregunta de salida). ¿Qué cambiarías de este notebook?

In [ ]:
# Tu turno

### Referencias
- Devlin et al. (2019), *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*, NAACL.
- Sanh et al. (2019), *DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter*.
- Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*, ICLR 2022.
- Maas et al. (2011), *Learning Word Vectors for Sentiment Analysis*, ACL (IMDB).
- Documentación de [Transformers](https://huggingface.co/docs/transformers) (`Trainer`, `pipeline`) y de [peft](https://huggingface.co/docs/peft).
- Material del curso del semestre anterior (`jdmartinev/si7011-deeplearning`, `Lecture06`): los notebooks de DistilBERT sobre IMDB.